<a href="https://colab.research.google.com/github/netsetos/agentic-ai-weekend-gcp-learners/blob/rag-production-hardening/module-06-function-calling/lesson-6.2-calling-loop/notebooks/GCP_Capstone_6.2_CallingLoop.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 6.2 Complete Calling Loop — LLM Decides → Code Executes → Result Returns → LLM Synthesizes
**Netsetos GenAI Engineering — GCP Capstone**

Build the production while-loop dispatcher, chain sequential calls, execute parallel calls, handle errors as data.


## Setup


In [ ]:
!pip install -q google-genai

from google.colab import auth
auth.authenticate_user()

PROJECT_ID = 'documind-ai-YOUR-ID'  # CHANGE

from google import genai
from google.genai import types

client = genai.Client(enterprise=True, project=PROJECT_ID,
                      location='global')
print(f'Client ready for {PROJECT_ID}')


## Cell 1: Define DocuMind Functions


In [ ]:
USD_INR = 85          # course-wide conversion rate
RATES = {"standard": 0.05, "priority": 0.12, "bulk": 0.03}

# DocuMind's demo corpus - the SAME five documents in every
# lesson of Modules 6, 7 and 8, so results stay comparable.
# 119 pages in total, which is what the cost examples bill.
GS = "gs://documind-acme"
_DOCS = [
    # doc_id, doc_type, page, score, quote
    ("hr_policy_2026", "policy", 12, 0.94,
     "A senior engineer serves a notice period of 60 days."),
    ("hr_policy_2026", "policy", 31, 0.81,
     "Earned leave is encashed on exit, capped at 45 days."),
    ("msa_acme_2026", "contract", 8, 0.88,
     "Either party may terminate on 90 days written notice."),
    ("inv_2026_0412", "invoice", 1, 0.76,
     "Total payable Rs 1,84,500, inclusive of 18% GST."),
    ("gstr1_q1_fy27", "form", 4, 0.68,
     "Outward taxable supplies for the quarter, GSTR-1."),
    ("rag_survey_2026", "research_paper", 6, 0.72,
     "Hybrid retrieval mixes dense and sparse signals."),
]
CORPUS = [{"chunk_id": f"{d}#{p}", "doc_type": t, "page": p,
           "source_uri": f"{GS}/{d}.pdf", "quote": q,
           "score": s} for d, t, p, s, q in _DOCS]
CITATION_FIELDS = ("chunk_id", "source_uri", "page", "quote",
                   "score")

# Document lengths, for the tool-chaining demos: retrieve
# finds chunks, and the cost tool bills whole documents.
DOC_PAGES = {"hr_policy_2026": 48, "msa_acme_2026": 32,
             "inv_2026_0412": 3, "gstr1_q1_fy27": 12,
             "rag_survey_2026": 24}          # 119 pages


def docs_of(citations: list) -> dict:
    """Distinct source documents behind a set of citations."""
    return {c["chunk_id"].split("#")[0]: True
            for c in citations}

USD_INR = 85   # course-wide conversion rate

def retrieve(query: str, doc_type: str = "all",
             top_k: int = 5) -> dict:
    """Retrieve grounded passages from DocuMind's corpus.

    Args:
        query: The question, in natural language
        doc_type: policy, contract, invoice, form,
            research_paper, or all
        top_k: How many passages to return
    """
    # A mock, but not a stub: it really filters and ranks, so
    # a question the corpus cannot answer returns NOTHING and
    # answerable=False. A mock that always succeeds teaches
    # that retrieval always succeeds - the one thing it never
    # does.
    words = {w for w in query.lower().split() if len(w) > 3}
    hits = [c for c in CORPUS
            if doc_type in ("all", c["doc_type"])
            and any(w in c["quote"].lower() for w in words)]
    hits.sort(key=lambda c: -c["score"])
    hits = hits[:top_k]
    top = hits[0]["score"] if hits else 0.0
    return {
        "citations": [{k: c[k] for k in CITATION_FIELDS}
                      for c in hits],
        "answerable": bool(hits),
        "confidence": ("high" if top >= 0.85 else
                       "medium" if hits else "low"),
    }

def calculate_processing_cost(
        total_pages: int, num_documents: int = 1,
        processing_type: str = "standard") -> dict:
    """Estimate document processing cost in USD and INR.

    Args:
        total_pages: Total page count across all documents
        num_documents: How many documents those pages span
        processing_type: standard, priority, or bulk
    """
    rate = RATES.get(processing_type, RATES["standard"])
    cost = total_pages * rate
    return {"num_documents": num_documents,
            "total_pages": total_pages,
            "processing_type": processing_type,
            "rate_per_page": rate,
            "cost_usd": round(cost, 2),
            "cost_inr": round(cost * USD_INR, 2)}

def get_usage_stats(metric: str, days: int = 7) -> dict:
    """Get DocuMind pipeline usage statistics.

    Args:
        metric: queries, costs, latency, or users
        days: Number of days to look back
    """
    mock = {"queries": 1247, "costs": 18.50,
            "latency": 245, "users": 42}
    return {"metric": metric, "period": f"last {days} days",
            "value": mock.get(metric, 0), "trend": "+12%"}

FUNCTIONS = {
    'retrieve': retrieve,
    'calculate_processing_cost': calculate_processing_cost,
    'get_usage_stats': get_usage_stats,
}
print(f'Registered {len(FUNCTIONS)} functions')


## Cell 2: The While-Loop Dispatcher


In [ ]:
def run_function_loop(client, prompt, tools, functions=None, dispatch=None,
                      model='gemini-3.6-flash',
                      system_instruction='', max_turns=10):
    """Production while-loop: call until model returns text.

    Pass EITHER functions= (a name -> callable dict, the simple case) OR
    dispatch= (a callable(name, args) -> dict). dispatch is the seam the
    FunctionRegistry below plugs into: call the callables directly and every
    guard the registry enforces - blocked list, timeouts, logging - is skipped.
    """
    if dispatch is None:
        if functions is None:
            raise ValueError('pass functions= or dispatch=')

        def dispatch(name, args):
            return {'result': functions[name](**args)}

    # Disable Automatic Function Calling: WE dispatch each call here.
    # With raw callables in tools and AFC on (the default), the SDK
    # runs the whole loop itself and response.function_calls is empty.
    config = types.GenerateContentConfig(
        tools=tools, system_instruction=system_instruction,
        automatic_function_calling=types.AutomaticFunctionCallingConfig(
            disable=True))
    contents = [types.Content(role='user', parts=[
        types.Part.from_text(text=prompt)])]

    for turn in range(max_turns):
        response = client.models.generate_content(
            model=model, contents=contents, config=config)
        contents.append(response.candidates[0].content)

        if not response.function_calls:
            return response.text  # Done!

        result_parts = []
        for fc in response.function_calls:
            print(f'  [Turn {turn+1}] {fc.name}({dict(fc.args)})')
            try:
                payload = dispatch(fc.name, fc.args)
                result_parts.append(
                    types.Part.from_function_response(
                        name=fc.name, response=payload))
            except Exception as e:
                result_parts.append(
                    types.Part.from_function_response(
                        name=fc.name, response={'error': str(e)}))

        contents.append(types.Content(role='user', parts=result_parts))

    return 'Max turns reached.'

print('While-loop dispatcher ready')

## Cell 3: Single-Call Test


In [ ]:
# Simple single-call test
TOOLS = [retrieve, calculate_processing_cost, get_usage_stats]

answer = run_function_loop(
    client=client,
    prompt='How many queries did we get this week?',
    tools=TOOLS,
    functions=FUNCTIONS)
print('\n=== Answer ===')
print(answer)


## Cell 4: Sequential Chain Test


In [ ]:
# Sequential: search → then calculate cost based on results
answer = run_function_loop(
    client=client,
    prompt='Find all legal documents and estimate bulk processing cost',
    tools=TOOLS,
    functions=FUNCTIONS,
    system_instruction='You are DocuMind AI. When estimating costs, first '
                       'search for documents to get accurate page counts. '
                       'Never guess page numbers. If search returns no '
                       'results, explain this clearly.')
print('\n=== Sequential Chain Answer ===')
print(answer)


## Cell 5: Automatic Chat Session


In [ ]:
# Chat session with auto function calling
chat = client.chats.create(
    model='gemini-3.6-flash',
    config=types.GenerateContentConfig(
        tools=TOOLS,
        system_instruction='You are DocuMind AI. Use tools to answer '
                           'document questions accurately.'))

# Turn 1: search
r1 = chat.send_message('What legal documents do we have?')
print(f'Turn 1: {r1.text[:150]}...' if len(r1.text) > 150 else f'Turn 1: {r1.text}')

# Turn 2: follow-up cost (uses context from turn 1)
r2 = chat.send_message('How much would priority processing cost for those?')
print(f'\nTurn 2: {r2.text[:150]}...' if len(r2.text) > 150 else f'\nTurn 2: {r2.text}')

# Turn 3: different tool
r3 = chat.send_message('Show me this month\'s query stats')
print(f'\nTurn 3: {r3.text[:150]}...' if len(r3.text) > 150 else f'\nTurn 3: {r3.text}')

# View history
print(f'\nTotal history messages: {len(chat.get_history())}')


## Cell 6: FunctionRegistry with Timeouts


In [ ]:
from concurrent.futures import ThreadPoolExecutor, TimeoutError as FuturesTimeout
import logging

class FunctionRegistry:
    BLOCKED = {'delete_document', 'send_email', 'modify_access'}

    def __init__(self):
        self._funcs = {}
        self._timeouts = {}

    def register(self, name, func, timeout=30):
        self._funcs[name] = func
        self._timeouts[name] = timeout

    def execute(self, name, args):
        if name in self.BLOCKED:
            return {'error': f'{name} requires manual approval'}
        if name not in self._funcs:
            return {'error': f'Unknown function: {name}'}
        try:
            with ThreadPoolExecutor(max_workers=1) as pool:
                future = pool.submit(self._funcs[name], **args)
                result = future.result(timeout=self._timeouts[name])
            return {'result': result}
        except FuturesTimeout:
            return {'error': f'{name} timed out after {self._timeouts[name]}s'}
        except Exception as e:
            return {'error': f'Failed: {str(e)}'}

    def tools(self):
        """The registered callables, for the SDK to build declarations from."""
        return list(self._funcs.values())

# Test
reg = FunctionRegistry()
reg.register('retrieve', retrieve, timeout=30)
reg.register('calculate_processing_cost', calculate_processing_cost, timeout=10)
reg.register('get_usage_stats', get_usage_stats, timeout=60)

print('Registry tests:')
print(f"  search: {reg.execute('retrieve', {'query': 'test'})}")
print(f"  blocked: {reg.execute('delete_document', {'id': 'msa_acme_2026'})}")
print(f"  unknown: {reg.execute('nonexistent', {})}")
print(f"  bad args: {reg.execute('calculate_processing_cost', {'invalid': True})}")


## Cell 7: DocuMindAgent Class


In [ ]:
from datetime import date


class DocuMindAgent:
    def __init__(self, client):
        self.client = client
        self.registry = FunctionRegistry()
        for name, fn, timeout in (
                ('retrieve', retrieve, 30),
                ('calculate_processing_cost', calculate_processing_cost, 10),
                ('get_usage_stats', get_usage_stats, 60)):
            self.registry.register(name, fn, timeout=timeout)

    def system_prompt(self):
        # Computed per call, not stored as a class attribute. A class attribute is
        # evaluated once at import, so a service that stays up for three weeks would
        # keep telling the model it is still the day it booted.
        return ('You are DocuMind AI, a document intelligence assistant. '
                f'Today is {date.today().isoformat()}. Use tools for document questions. '
                'When estimating costs, first search for real document counts. '
                'If search returns no results, explain clearly.')

    def ask(self, question):
        # dispatch=, not functions=. This is the whole point of the registry: the
        # blocked list, the per-tool timeout and the logging only run if the loop
        # goes through execute(). Passing functions= here would skip all three.
        return run_function_loop(
            client=self.client, prompt=question,
            tools=self.registry.tools(), dispatch=self.registry.execute,
            system_instruction=self.system_prompt(), max_turns=8)

    def create_chat(self):
        # NOTE the asymmetry, and know which path you are on. A chat session with
        # Automatic Function Calling left on hands dispatch to the SDK, which calls
        # your callables directly - the registry's guards do NOT apply here. If you
        # need them on a multi-turn conversation, disable AFC and drive the loop.
        return self.client.chats.create(
            model='gemini-3.6-flash',
            config=types.GenerateContentConfig(
                tools=self.registry.tools(),
                system_instruction=self.system_prompt()))


# Test single-turn
agent = DocuMindAgent(client)
print('=== Single-turn ===')
print(agent.ask('Find all legal documents and estimate standard processing cost'))

# The guard is real: ask for something on the blocked list and the loop never
# reaches the function.
print('\n=== Blocked call ===')
print(agent.registry.execute('delete_document', {'id': 'msa_acme_2026'}))

# Test multi-turn
print('\n=== Multi-turn ===')
chat = agent.create_chat()
print(chat.send_message('What invoices do we have?').text)
print(chat.send_message('How much to process them in bulk?').text)

## ✅ Lesson 6.2 Complete!

- ✅ While-loop dispatcher (loop until text)
- ✅ Conversation history with thought_signature preservation
- ✅ Sequential chaining (search → calculate using results)
- ✅ Parallel execution with id-based result mapping
- ✅ FunctionRegistry with timeouts and destructive-op blocking
- ✅ Error-as-data pattern (model explains failures naturally)
- ✅ Chat sessions with auto function calling
- ✅ Google Search + custom tools combined
- ✅ DocuMindAgent class (single-turn + multi-turn)

**Next: Lesson 6.3 — Google Search Grounding & Built-in Tools**
